# Representation & mechanism diagnostics — PIC held-out (Vanilla / Kernel-age)

Compares **frozen** encoders on the **held-out PIC split only**. For each patient we extract two
representations and never touch logits for the representation analysis:

- `h_encoder` — encoder output at the last event, **before** demographic/static-feature concat.
- `h_head_input` — `cat([h_encoder, demo_last])`, the vector fed **into** the prediction head.

Then: age-band probes + neighborhood diagnostics (balanced & unbalanced), UMAP visualizations,
Kernel-age mechanism analysis, and a counterfactual age sweep.

**Guardrails**: no training; held-out split only; disease labels never used to fit UMAP or probes'
features; train patients never loaded. All project-specific bits live in the `TODO` cells.

Only **Vanilla** and **Kernel-age** have PIC-fine-tuned checkpoints on disk, so they run out of the
box. **Additive-age / Random-constant / Shuffled-age** are disabled registry entries with a short
TODO loader (enable once you have a checkpoint).

In [ ]:
# Dependencies: torch, numpy, pandas, matplotlib, scikit-learn, umap-learn
# Install if needed:  %pip install -q scikit-learn umap-learn
import importlib
_req = {"numpy":"numpy","pandas":"pandas","torch":"torch","matplotlib":"matplotlib",
        "sklearn":"scikit-learn","umap":"umap-learn"}
_missing = [pkg for mod,pkg in _req.items() if importlib.util.find_spec(mod) is None]
print("Install:  %pip install -q " + " ".join(_missing) if _missing else "All packages importable.")

## CONFIG

Edit paths/flags here. Defaults match this repo.

In [ ]:
from pathlib import Path

REPO_ROOT = Path("/home/suraj/Git/Age-conditioned-pediatric-EHR")  # <-- VERIFY

PIC_TASK = "heart_malformations"  # PIC congenital heart malformation task
PIC_TEST_SPLIT_DIR = REPO_ROOT / "data" / "tensorized" / "pic" / PIC_TASK / "test"  # held-out only
PIC_VOCAB = REPO_ROOT / "data" / "processed" / "pic" / "code_vocab_pic.json"
PIC_EMB   = REPO_ROOT / "data" / "processed" / "pic" / "bge_embeddings_pic.pt"

VANILLA_PRETRAIN = REPO_ROOT / "checkpoints" / "run_20260427_152603" / "best_pretrain.pt"
AGE_PRETRAIN     = REPO_ROOT / "checkpoints" / "age_real_202605112156" / "epoch_012.pt"

_FT = REPO_ROOT / "checkpoints" / "finetune" / "PIC"
# kind: "pic_finetune" (loads best.pt) | "shuffled_age" (control over a base) | "ablation" (TODO)
MODEL_REGISTRY = {
    "Vanilla":         dict(kind="pic_finetune", backbone="vanilla", ckpt_dir=_FT/(PIC_TASK+"_vanilla"), enabled=True),
    "Kernel-age":      dict(kind="pic_finetune", backbone="age",     ckpt_dir=_FT/(PIC_TASK+"_age"),     enabled=True),
    "Additive-age":    dict(kind="ablation", arm="additive",         ckpt_dir=None, enabled=False),  # <-- TODO ckpt
    "Random-constant": dict(kind="ablation", arm="random_constant",  ckpt_dir=None, enabled=False),  # optional
    "Shuffled-age":    dict(kind="shuffled_age", base="Kernel-age",  enabled=False),                 # optional control
}

REP_TYPES = ["h_encoder", "h_head_input"]

AGE_BANDS = [("<1",0.0,1.0), ("1-5",1.0,6.0), ("6-11",6.0,12.0), ("12-17",12.0,18.0)]
BAND_ORDER = [b[0] for b in AGE_BANDS]

BATCH_SIZE, MAX_SEQ_LEN, NUM_WORKERS, DEVICE = 128, 1024, 0, "cuda"

RANDOM_STATE = 42
USE_PCA, PCA_DIM = True, 50
UMAP_KW = dict(metric="cosine", n_neighbors=30, min_dist=0.1, random_state=RANDOM_STATE)

DIAG_METRIC = "cosine"
KNN_K = 15
N_BALANCED_SEEDS = 50
N_BOOTSTRAP = 1000
PROBE_TEST_FRAC = 0.3

MECH_MODEL = "Kernel-age"
KERNEL_AGES_YR = [0.1, 0.5, 1.0, 3.0, 5.0, 10.0, 17.0]
LAG_BUCKET_EDGES_DAYS = [0, 1, 7, 30, 90, 365, 100000]
LAG_BUCKET_LABELS = ["<1d","1-7d","1-4wk","1-3mo","3-12mo",">1yr"]
CF_N_PATIENTS = 64
CF_SWEEP_AGES_YR = [0.1, 0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 17.0]

OUT_DIR = REPO_ROOT / "outputs" / "representation_analysis"
FIG_DIR, TBL_DIR = OUT_DIR/"figures", OUT_DIR/"tables"
for d in (OUT_DIR, FIG_DIR, TBL_DIR): d.mkdir(parents=True, exist_ok=True)

MODELS = {k:v for k,v in MODEL_REGISTRY.items() if v.get("enabled")}
print("Enabled:", list(MODELS), "| OUT_DIR:", OUT_DIR)
assert "train" not in str(PIC_TEST_SPLIT_DIR).lower(), "Refusing: split looks like TRAIN."
assert PIC_TEST_SPLIT_DIR.exists(), PIC_TEST_SPLIT_DIR

In [ ]:
import json, warnings, math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
import umap

np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)
device = torch.device(DEVICE if (DEVICE=="cuda" and torch.cuda.is_available()) else "cpu")
print("device:", device)

plt.rcParams.update({"figure.dpi":120,"savefig.dpi":300,"font.size":11,"axes.titlesize":12,
                     "axes.spines.top":False,"axes.spines.right":False})
BAND_COLORS = {"<1":"#0173B2","1-5":"#029E73","6-11":"#DE8F05","12-17":"#CC3311"}

def savefig(fig, name):
    for ext in ("png","svg"): fig.savefig(FIG_DIR/(name+"."+ext), bbox_inches="tight")
    print("fig ->", name+".png/.svg")

def savetable(df, name):
    df.to_csv(TBL_DIR/(name+".csv"), index=False); print("csv ->", name+".csv"); return df

## TODO (1/2) — project imports

All repo-specific imports isolated here.

In [ ]:
import sys
if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))

from finetune.dataset import (TensorizedDiseaseClassificationDataset, disease_collate,
                              _dataloader_worker_init)
from finetune.PIC.pic_age_eval_common import load_finetuned_classifier
import finetune.PIC.pic_age_eval_common as _ec
# point the loader's module-level paths at our CONFIG
_ec.AGE_PRETRAIN, _ec.VANILLA_PRETRAIN = AGE_PRETRAIN, VANILLA_PRETRAIN
_ec.PIC_EMB, _ec.PIC_VOCAB = PIC_EMB, PIC_VOCAB
print("project imports OK")

## TODO (2/2) — frozen encoders + dual representation extractor

For the PIC classifier the head is `classifier(cat([h_encoder, demo_last]))`, so:
`h_encoder = h_repr[last]` and `h_head_input = cat([h_encoder, demo_last])`. Re-implement
`build_model` if your API differs. Ablation arms (Additive/Random) are a short TODO stub.

In [ ]:
from dataclasses import dataclass
from typing import Callable, Optional

def _last_idx(batch):
    return (batch["attention_mask"].sum(1).long() - 1).clamp(min=0)

def _dual_repr(backbone, batch):
    out = backbone(batch, return_repr_only=True)
    h, d = out["h_repr"], out["demo_features"]
    li = _last_idx(batch); r = torch.arange(h.shape[0], device=h.device)
    h_enc, d_last = h[r, li], d[r, li]
    return {"h_encoder": h_enc, "h_head_input": torch.cat([h_enc, d_last], -1)}

@dataclass
class LoadedModel:
    name: str; kind: str; model: torch.nn.Module
    extract_fn: Callable; predict_fn: Optional[Callable] = None

def _shuffle_age(batch, seed=RANDOM_STATE):
    # control: permute each patient's absolute age level, keep within-seq deltas
    b = dict(batch); demo = b["demographics"].clone(); mask = b["attention_mask"]
    li = _last_idx(b); r = torch.arange(demo.shape[0], device=demo.device)
    age_idx = demo[r, li, 0].clone()
    g = torch.Generator(); g.manual_seed(seed)
    perm = torch.randperm(age_idx.shape[0], generator=g).to(demo.device)
    demo[..., 0] = (demo[..., 0] + (age_idx[perm]-age_idx).view(-1,1)).clamp(min=0.0) * mask.to(demo.dtype)
    b["demographics"] = demo; return b

def build_model(name, spec, cache):
    kind = spec["kind"]
    if kind == "pic_finetune":
        m = load_finetuned_classifier(Path(spec["ckpt_dir"]), backbone=spec["backbone"], device=device).eval()
        for p in m.parameters(): p.requires_grad = False
        return LoadedModel(name, kind, m, lambda b: _dual_repr(m.backbone, b),
                           lambda b: torch.sigmoid(m(b)))
    if kind == "shuffled_age":
        base = cache.get(spec["base"])
        if base is None: raise RuntimeError("shuffled_age needs base '%s' loaded first" % spec["base"])
        m = base.model
        return LoadedModel(name, kind, m, lambda b: _dual_repr(m.backbone, _shuffle_age(b)),
                           lambda b: torch.sigmoid(m(_shuffle_age(b))))
    if kind == "ablation":
        # TODO: model_ablation arms (Additive-age / Random-constant). No PIC-fine-tuned
        # ablation checkpoint exists on disk, and the ablation collate differs
        # (expects batch["age_years"], demo_dim=2). To enable once you have one:
        #   from model_ablation.model_finetune import TALEEHRAblationClassifier
        #   from model_ablation.dataset_finetune import disease_collate as abl_collate
        #   m = TALEEHRAblationClassifier(arm=spec["arm"], pretrained_ckpt_path=<pretrain>,
        #                                 embedding_path=PIC_EMB)
        #   m.load_state_dict(torch.load(Path(spec["ckpt_dir"])/"best.pt")["model_state_dict"])
        #   m.to(device).eval(); freeze; register abl_collate in COLLATE_BY_KIND;
        #   extract_fn = lambda b: _dual_repr(m.backbone, b)
        raise NotImplementedError("Ablation loader for '%s' is a TODO (no PIC ckpt on disk)." % name)
    raise ValueError("unknown kind: %s" % kind)

COLLATE_BY_KIND = {"pic_finetune": disease_collate, "shuffled_age": disease_collate}
print("loaders ready")

## Age bands, dataloader, metadata

In [ ]:
from torch.utils.data import DataLoader

def assign_band(age):
    age = np.asarray(age, float); out = np.full(age.shape, None, dtype=object)
    for n,lo,hi in AGE_BANDS: out[(age>=lo)&(age<hi)] = n
    return out

def make_loader(collate=disease_collate):
    ds = TensorizedDiseaseClassificationDataset(PIC_TEST_SPLIT_DIR, max_seq_len=MAX_SEQ_LEN)
    return ds, DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate,
                          num_workers=NUM_WORKERS,
                          worker_init_fn=_dataloader_worker_init if NUM_WORKERS>0 else None,
                          pin_memory=(device.type=="cuda"))

def move(b):
    return {k:(v.to(device) if isinstance(v, torch.Tensor) else v) for k,v in b.items()}

def batch_meta(batch):
    li = _last_idx(batch); r = torch.arange(len(li), device=batch["demographics"].device)
    age = batch["demographics"][r, li, 0].detach().cpu().numpy().astype(np.float64)
    subj = batch.get("subject_id")
    pid = np.asarray([int(x) for x in subj], dtype=np.int64)
    lab = batch["labels"].detach().cpu().numpy().astype(np.float64) if batch.get("labels") is not None else None
    return pid, age, lab

_ds, _ = make_loader(); print("held-out patients:", len(_ds)); del _ds

## Extract both representations for every model

Deterministic order keeps metadata aligned across models. Frozen, `eval()`, `no_grad`.

In [ ]:
loaded = {}
for name, spec in sorted(MODELS.items(), key=lambda kv: kv[1]["kind"]=="shuffled_age"):
    print("loading", name); loaded[name] = build_model(name, spec, loaded)

embeddings = {n:{r:[] for r in REP_TYPES} for n in loaded}
meta_ref = None
for name, lm in loaded.items():
    _, loader = make_loader(COLLATE_BY_KIND.get(lm.kind, disease_collate))
    reps = {r:[] for r in REP_TYPES}; pids=[]; ages=[]; labs=[]
    with torch.no_grad():
        for batch in loader:
            batch = move(batch)
            out = lm.extract_fn(batch)
            for r in REP_TYPES: reps[r].append(out[r].cpu().float().numpy())
            pid, age, lab = batch_meta(batch)  # metadata from real (unshuffled) batch
            pids.append(pid); ages.append(age)
            labs.append(lab if lab is not None else np.full(pid.shape, np.nan))
    for r in REP_TYPES: embeddings[name][r] = np.concatenate(reps[r])
    m = (np.concatenate(pids), np.concatenate(ages), np.concatenate(labs))
    if meta_ref is None: meta_ref = m
    else: assert np.array_equal(meta_ref[0], m[0]), "patient order mismatch: "+name
    if device.type=="cuda": torch.cuda.empty_cache()
    print(" ", name, {r:embeddings[name][r].shape for r in REP_TYPES})

## Pediatric filter + save embeddings/metadata

In [ ]:
pid_all, age_all, lab_all = meta_ref
bands = assign_band(age_all)
ped = np.array([b is not None for b in bands])
print("pediatric [0,18):", int(ped.sum()), "/", len(age_all))

patient_id, age_years = pid_all[ped], age_all[ped]
age_band = np.array([b for b in bands[ped]], dtype=object)
meta_df = pd.DataFrame({"patient_id":patient_id, "age_at_index":age_years, "age_band":age_band})
has_chm = lab_all is not None and not np.all(np.isnan(lab_all))
if has_chm: meta_df["congenital_heart_malformation"] = lab_all[ped]

emb = {n:{r:embeddings[n][r][ped] for r in REP_TYPES} for n in embeddings}
print(meta_df["age_band"].value_counts().reindex(BAND_ORDER).fillna(0).astype(int).to_string())

savetable(meta_df, "representation_metadata")
npz = {"patient_id":patient_id, "age_at_index":age_years, "age_band":age_band.astype("U16")}
if has_chm: npz["congenital_heart_malformation"] = meta_df["congenital_heart_malformation"].to_numpy()
for n in emb:
    for r in REP_TYPES:
        npz["H__%s__%s" % (n.replace(" ","_").replace("-","_"), r)] = emb[n][r].astype(np.float32)
np.savez_compressed(OUT_DIR/"representations.npz", **npz)
print("saved representations.npz")
meta_df.head()

## Diagnostic utilities

Computed on standardized (+PCA-50) `h`, **never** on UMAP coords. Probe = multinomial logistic
regression with `class_weight='balanced'`; kNN + silhouette use cosine distance.

In [ ]:
def preprocess(H, seed=RANDOM_STATE):
    X = StandardScaler().fit_transform(H)
    if USE_PCA:
        nc = min(PCA_DIM, X.shape[0]-1, X.shape[1])
        if nc >= 2: X = PCA(n_components=nc, random_state=seed).fit_transform(X)
    return X

def probe_eval(X, y, seed=RANDOM_STATE, do_ci=True):
    classes = np.unique(y)
    nan = dict(balanced_accuracy=np.nan, macro_f1=np.nan)
    if len(classes) < 2: return nan
    try:
        Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=PROBE_TEST_FRAC, random_state=seed, stratify=y)
        clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, ytr)
        yp = clf.predict(Xte)
    except Exception:
        return nan
    res = dict(balanced_accuracy=balanced_accuracy_score(yte,yp), macro_f1=f1_score(yte,yp,average="macro"))
    if do_ci:
        rng = np.random.default_rng(seed); ba=[]; mf=[]; n=len(yte)
        for _ in range(N_BOOTSTRAP):
            idx = rng.integers(0,n,n)
            if len(np.unique(yte[idx]))<2: continue
            ba.append(balanced_accuracy_score(yte[idx],yp[idx])); mf.append(f1_score(yte[idx],yp[idx],average="macro"))
        res["balanced_accuracy_lo"],res["balanced_accuracy_hi"] = (np.percentile(ba,[2.5,97.5]) if ba else (np.nan,np.nan))
        res["macro_f1_lo"],res["macro_f1_hi"] = (np.percentile(mf,[2.5,97.5]) if mf else (np.nan,np.nan))
    return res

def knn_metrics(X, y, ages, k=KNN_K, metric=DIAG_METRIC):
    n = X.shape[0]; keff = min(k, n-1)
    if keff < 1: return dict(knn_purity=np.nan, neighbor_age_mae=np.nan), {}
    _, idx = NearestNeighbors(n_neighbors=keff+1, metric=metric).fit(X).kneighbors(X)
    nb = idx[:,1:]; same = (y[nb]==y[:,None])
    per = {b: float(same[y==b].mean()) for b in BAND_ORDER if (y==b).sum()>0}
    return dict(knn_purity=float(same.mean()),
                neighbor_age_mae=float(np.abs(ages[nb]-ages[:,None]).mean())), per

def silhouette_band(X, y, metric=DIAG_METRIC):
    return float(silhouette_score(X, y, metric=metric)) if 2<=len(np.unique(y))<len(y) else np.nan

def balanced_idx(y, seed):
    rng = np.random.default_rng(seed)
    present = [b for b in BAND_ORDER if (y==b).sum()>0]
    per = min((y==b).sum() for b in present)
    out = np.concatenate([rng.choice(np.where(y==b)[0], per, replace=False) for b in present])
    rng.shuffle(out); return out, int(per)

## Diagnostics: unbalanced (full) + balanced (equal-per-band, 50 seeds)

In [ ]:
y_all = meta_df["age_band"].to_numpy()
ages_arr = meta_df["age_at_index"].to_numpy()
features = {(n,r): preprocess(emb[n][r]) for n in emb for r in REP_TYPES}

# ---- unbalanced ----
unb, perband = [], []
for n in emb:
    for r in REP_TYPES:
        X = features[(n,r)]
        pr = probe_eval(X, y_all); km, pb = knn_metrics(X, y_all, ages_arr)
        unb.append(dict(model=n, representation=r, analysis="unbalanced", n=len(y_all),
                        **pr, **km, silhouette=silhouette_band(X,y_all)))
        for b,v in pb.items(): perband.append(dict(model=n, representation=r, analysis="unbalanced", band=b, knn_purity=v))
unb_df = savetable(pd.DataFrame(unb), "diagnostics_unbalanced")

# ---- balanced ----
seeds = [RANDOM_STATE+i for i in range(N_BALANCED_SEEDS)]
_, per_n = balanced_idx(y_all, RANDOM_STATE)
print("balanced per-band N =", per_n, "x", len([b for b in BAND_ORDER if (y_all==b).any()]), "bands x", N_BALANCED_SEEDS, "seeds")
bal, bal_pb = [], []
def ms(a): a=np.asarray(a,float); return float(np.nanmean(a)), float(np.nanstd(a))
for n in emb:
    for r in REP_TYPES:
        Xf = features[(n,r)]
        acc=[]; f1=[]; pur=[]; mae=[]; sil=[]; pbacc={b:[] for b in BAND_ORDER}
        for s in seeds:
            i,_ = balanced_idx(y_all, s); X=Xf[i]; y=y_all[i]; ag=ages_arr[i]
            pr = probe_eval(X, y, seed=s, do_ci=False); km, pb = knn_metrics(X, y, ag)
            acc.append(pr["balanced_accuracy"]); f1.append(pr["macro_f1"])
            pur.append(km["knn_purity"]); mae.append(km["neighbor_age_mae"]); sil.append(silhouette_band(X,y))
            for b,v in pb.items(): pbacc[b].append(v)
        rec = dict(model=n, representation=r, analysis="balanced", per_band_n=per_n, n_seeds=N_BALANCED_SEEDS)
        for lbl,a in [("balanced_accuracy",acc),("macro_f1",f1),("knn_purity",pur),
                      ("neighbor_age_mae",mae),("silhouette",sil)]:
            rec[lbl+"_mean"], rec[lbl+"_std"] = ms(a)
        bal.append(rec)
        for b in BAND_ORDER:
            if pbacc[b]: bal_pb.append(dict(model=n, representation=r, analysis="balanced", band=b,
                                            knn_purity_mean=float(np.nanmean(pbacc[b])), knn_purity_std=float(np.nanstd(pbacc[b]))))
bal_df = savetable(pd.DataFrame(bal), "diagnostics_balanced")
savetable(pd.DataFrame(perband+bal_pb), "knn_purity_per_band")
bal_df

## UMAP embeddings (fit on `h` only — labels never used)

In [ ]:
umap_emb, umap_bal = {}, {}
bal_i, _ = balanced_idx(y_all, RANDOM_STATE)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for n in emb:
        for r in REP_TYPES:
            umap_emb[(n,r)] = umap.UMAP(n_components=2, **UMAP_KW).fit_transform(features[(n,r)])
            umap_bal[(n,r)] = umap.UMAP(n_components=2, **UMAP_KW).fit_transform(features[(n,r)][bal_i])
            print("UMAP", n, r)

## UMAP plots per model: age band, continuous age, CHM, balanced band

In [ ]:
def _axes(nc):
    fig, ax = plt.subplots(1, nc, figsize=(5.0*nc, 5.2), squeeze=False); return fig, ax[0]

def _clean(ax, title):
    ax.set_title(title, fontweight="bold"); ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal", adjustable="datalim")

names = list(emb)

def plot_band(rep):
    fig, axes = _axes(len(names)); present=[b for b in BAND_ORDER if (meta_df.age_band==b).any()]
    counts = meta_df.age_band.value_counts()
    for ax,n in zip(axes,names):
        Z = umap_emb[(n,rep)]
        for b in present:
            m=(meta_df.age_band==b).to_numpy()
            ax.scatter(Z[m,0],Z[m,1],s=6,alpha=.55,linewidths=0,color=BAND_COLORS[b],rasterized=True)
        _clean(ax,n)
    h=[Line2D([0],[0],marker="o",linestyle="",markerfacecolor=BAND_COLORS[b],markeredgecolor="none",
       markersize=8,label="%s (n=%d)"%(b,int(counts.get(b,0)))) for b in present]
    fig.legend(handles=h,title="Age band",loc="lower center",ncol=len(h),frameon=False,bbox_to_anchor=(.5,-.02))
    fig.suptitle("UMAP of %s by age band"%rep,y=1.02); fig.tight_layout(); return fig

def plot_age(rep):
    fig, axes=_axes(len(names)); a=meta_df.age_at_index.to_numpy(); sc=None
    for ax,n in zip(axes,names):
        Z=umap_emb[(n,rep)]
        sc=ax.scatter(Z[:,0],Z[:,1],c=a,s=6,alpha=.7,linewidths=0,cmap="viridis",vmin=0,vmax=18,rasterized=True)
        _clean(ax,n)
    fig.colorbar(sc,ax=axes,fraction=.025,pad=.02).set_label("age at index (yr)")
    fig.suptitle("UMAP of %s by continuous age"%rep,y=1.02); return fig

def plot_chm(rep):
    if not has_chm: return None
    fig, axes=_axes(len(names)); lab=meta_df.congenital_heart_malformation.to_numpy()
    for ax,n in zip(axes,names):
        Z=umap_emb[(n,rep)]; pos=lab>=.5
        ax.scatter(Z[~pos,0],Z[~pos,1],s=5,alpha=.3,linewidths=0,color="#BBBBBB",label="CHM-",rasterized=True)
        ax.scatter(Z[pos,0],Z[pos,1],s=8,alpha=.7,linewidths=0,color="#CC3311",label="CHM+",rasterized=True)
        _clean(ax,n)
    axes[-1].legend(frameon=False,loc="best")
    fig.suptitle("UMAP of %s by CHM (labels NOT used to fit UMAP)"%rep,y=1.02); fig.tight_layout(); return fig

def plot_balanced(rep):
    fig, axes=_axes(len(names)); yb=y_all[bal_i]; present=[b for b in BAND_ORDER if (yb==b).any()]
    for ax,n in zip(axes,names):
        Z=umap_bal[(n,rep)]
        for b in present:
            m=(yb==b); ax.scatter(Z[m,0],Z[m,1],s=8,alpha=.6,linewidths=0,color=BAND_COLORS[b],rasterized=True)
        _clean(ax,n)
    h=[Line2D([0],[0],marker="o",linestyle="",markerfacecolor=BAND_COLORS[b],markeredgecolor="none",markersize=8,label=b) for b in present]
    fig.legend(handles=h,title="Age band (balanced, n=%d each)"%int((yb==present[0]).sum()),
               loc="lower center",ncol=len(h),frameon=False,bbox_to_anchor=(.5,-.02))
    fig.suptitle("Balanced age-band UMAP of %s"%rep,y=1.02); fig.tight_layout(); return fig

for rep in REP_TYPES:
    savefig(plot_band(rep), "umap_%s_by_age_band"%rep); plt.show()
    savefig(plot_age(rep), "umap_%s_by_continuous_age"%rep); plt.show()
    f=plot_chm(rep);
    if f is not None: savefig(f, "umap_%s_by_chm"%rep); plt.show()
    savefig(plot_balanced(rep), "umap_%s_balanced_age_band"%rep); plt.show()

## Kernel-age mechanism: `w(tau|age)` and `||delta_alpha(age)||`

In [ ]:
mech = loaded.get(MECH_MODEL)
if mech is None:
    print("Kernel-age not loaded; skipping mechanism + counterfactual sections.")
else:
    ta = mech.model.backbone.time_aware_attention
    tw, age_emb = ta.temporal_weight, ta.age_emb
    age_cond = hasattr(tw, "age_coeff_gen") and getattr(tw.age_coeff_gen, "mode", "none") != "none"
    print("mechanism model:", MECH_MODEL, "| age-conditioned attention:", age_cond)

if mech is not None and age_cond:
    taus = np.unique(np.concatenate([[0.0], np.logspace(0, np.log10(3*365), 60)]))  # days
    dt = torch.tensor(np.log1p(taus/7.0), dtype=torch.float32).view(1,-1).to(device)
    fig, ax = plt.subplots(figsize=(7,4.5)); wt = {"lag_days": taus}
    with torch.no_grad():
        for a in KERNEL_AGES_YR:
            af = age_emb(torch.tensor([float(a)]).clamp(min=0).to(device))     # [1, age_dim]
            w = torch.sigmoid(tw.poly_value(dt, af)).reshape(-1).cpu().numpy() # [T]
            ax.plot(taus, w, label="%.2g yr"%a); wt["w_age_%.3g"%a]=w
    ax.set_xscale("log"); ax.set_xlabel("lag tau (days, log)"); ax.set_ylabel("w(tau | age)")
    ax.set_title("Kernel-age: learned temporal kernel by age"); ax.legend(frameon=False, fontsize=8, title="age")
    savefig(fig, "kernel_w_tau_by_age"); plt.show(); savetable(pd.DataFrame(wt), "kernel_w_tau_by_age")

    grid = np.linspace(0,18,181)
    with torch.no_grad():
        norms = np.array([float(np.linalg.norm(tw.age_coeff_gen(age_emb(
            torch.tensor([float(a)]).clamp(min=0).to(device))).reshape(-1).cpu().numpy())) for a in grid])
    fig, ax = plt.subplots(figsize=(7,4)); ax.plot(grid, norms, color="#333")
    for _,lo,hi in AGE_BANDS: ax.axvspan(lo, min(hi,18), alpha=.06)
    ax.set_xlabel("age (yr)"); ax.set_ylabel("|| delta_alpha(age) ||"); ax.set_title("Kernel-age: age-conditioning strength")
    savefig(fig, "kernel_delta_alpha_norm_by_age"); plt.show()
    dfn = pd.DataFrame({"age":grid, "delta_alpha_norm":norms}); dfn["age_band"]=assign_band(dfn.age.to_numpy())
    summ = dfn.dropna(subset=["age_band"]).groupby("age_band")["delta_alpha_norm"].agg(["mean","std","min","max"]).reindex(BAND_ORDER).reset_index()
    savetable(summ, "kernel_delta_alpha_by_band"); print(summ.to_string(index=False))

## Attention mass over lag buckets by age band (Vanilla vs Kernel-age)

In [ ]:
def attn_last_row(backbone, batch):
    ta = backbone.time_aware_attention
    ce = backbone.embedding_table[batch["code_indices"]]
    q,k,v = ta.mlp_q(ce), ta.mlp_k(ce), ta.mlp_v(ce)
    scores = torch.bmm(q, k.transpose(-1,-2)) * (1.0/math.sqrt(ta.d_model))
    dt = batch["delta_t"]
    ac = hasattr(ta.temporal_weight,"age_coeff_gen") and getattr(ta.temporal_weight.age_coeff_gen,"mode","none")!="none"
    if ac:
        poly = ta.temporal_weight.poly_value(dt, ta.age_emb(batch["demographics"][...,0].clamp(min=0.0)))
    else:
        poly = ta.temporal_weight.poly_value(dt)
    scores = scores*torch.sigmoid(poly) if ta.kernel_injection=="multiplicative" else scores+F.logsigmoid(poly)
    am = batch["attention_mask"]; L = scores.shape[1]
    causal = torch.tril(torch.ones((L,L), device=scores.device, dtype=torch.bool))
    full = (am.unsqueeze(1)&am.unsqueeze(2)) & causal.unsqueeze(0)
    attn = torch.softmax(scores.masked_fill(~full, float("-inf")), -1).masked_fill(~full, 0.0)
    li = _last_idx(batch); r = torch.arange(attn.shape[0], device=attn.device)
    t = batch["timestamps_days"]
    return attn[r,li], (t[r,li].unsqueeze(1)-t).abs(), am

if mech is not None:
    edges = np.array(LAG_BUCKET_EDGES_DAYS, float)
    def lag_mass(backbone):
        _, loader = make_loader(disease_collate)
        acc={b:np.zeros(len(LAG_BUCKET_LABELS)) for b in BAND_ORDER}; cnt={b:0 for b in BAND_ORDER}
        with torch.no_grad():
            for batch in loader:
                batch = move(batch)
                al, lag, am = attn_last_row(backbone, batch)
                li=_last_idx(batch); r=torch.arange(am.shape[0],device=am.device)
                bnd = assign_band(batch["demographics"][r,li,0].cpu().numpy())
                al=al.cpu().numpy(); lag=lag.cpu().numpy(); amn=am.cpu().numpy()
                bi=np.digitize(lag, edges[1:-1])
                for i in range(al.shape[0]):
                    b=bnd[i]
                    if b is None: continue
                    for j in range(len(LAG_BUCKET_LABELS)): acc[b][j]+=al[i][(bi[i]==j)&amn[i]].sum()
                    cnt[b]+=1
        rows=[]
        for b in BAND_ORDER:
            if cnt[b]:
                for j,l in enumerate(LAG_BUCKET_LABELS):
                    rows.append(dict(band=b,lag_bucket=l,mean_attention_mass=float(acc[b][j]/cnt[b]),n_patients=cnt[b]))
        return pd.DataFrame(rows)

    tabs={n:lag_mass(loaded[n].model.backbone).assign(model=n) for n in ("Vanilla",MECH_MODEL) if n in loaded}
    if tabs:
        savetable(pd.concat(tabs.values(), ignore_index=True), "attention_mass_by_lag_band")
        fig, axes = plt.subplots(1,len(tabs),figsize=(6.5*len(tabs),4.5),squeeze=False); axes=axes[0]
        x=np.arange(len(BAND_ORDER)); w=0.8/len(LAG_BUCKET_LABELS)
        cmap=plt.cm.viridis(np.linspace(.1,.9,len(LAG_BUCKET_LABELS)))
        for ax,(mn,t) in zip(axes, tabs.items()):
            for j,bk in enumerate(LAG_BUCKET_LABELS):
                vals=[t[(t.band==b)&(t.lag_bucket==bk)].mean_attention_mass.sum() for b in BAND_ORDER]
                ax.bar(x+j*w, vals, w, label=bk, color=cmap[j])
            ax.set_xticks(x+0.4-w/2); ax.set_xticklabels(BAND_ORDER); ax.set_title(mn,fontweight="bold")
            ax.set_xlabel("age band"); ax.set_ylabel("mean attention mass")
        axes[-1].legend(title="lag",frameon=False,fontsize=8,bbox_to_anchor=(1.02,1),loc="upper left")
        fig.suptitle("Attention mass over lag buckets by age band",y=1.03); fig.tight_layout()
        savefig(fig, "attention_mass_by_lag_band"); plt.show()

## Counterfactual age sweep (Kernel-age)

Fix a sample of patients' events & timestamps; sweep the age (ageing earlier events consistently).
Measure `||delta_alpha||`, `h_encoder` shift, attention lag distribution, and CHM probability.

In [ ]:
if mech is not None:
    kb, km = mech.model.backbone, mech.model
    _, loader = make_loader(disease_collate)
    base = None
    for batch in loader: base = move(batch); break
    n = min(CF_N_PATIENTS, base["attention_mask"].shape[0])
    base = {k:(v[:n] if isinstance(v,(torch.Tensor,list)) else v) for k,v in base.items()}
    t, am = base["timestamps_days"], base["attention_mask"]
    li=_last_idx(base); r=torch.arange(n, device=t.device); t_last=t[r,li]

    def set_age(a):
        b = dict(base); demo = b["demographics"].clone()
        new = (float(a)*365.25 - t_last.unsqueeze(1) + t)/365.25
        demo[...,0] = new.clamp(min=0.0)*am.to(demo.dtype); b["demographics"]=demo; return b

    edges=np.array(LAG_BUCKET_EDGES_DAYS,float)
    sweep=CF_SWEEP_AGES_YR; href=None; hshift=[]; chm=[]; dalpha=[]; lagm=[]
    ta=kb.time_aware_attention
    with torch.no_grad():
        for a in sweep:
            ba=set_age(a)
            he=_dual_repr(kb, ba)["h_encoder"].cpu().numpy()
            if href is None: href=he
            hshift.append(float(np.linalg.norm(he-href,axis=1).mean()))
            chm.append(float(torch.sigmoid(km(ba)).cpu().numpy().mean()))
            if hasattr(ta.temporal_weight,"age_coeff_gen"):
                af=ta.age_emb(torch.tensor([float(a)]).clamp(min=0).to(device))
                dalpha.append(float(np.linalg.norm(ta.temporal_weight.age_coeff_gen(af).reshape(-1).cpu().numpy())))
            else: dalpha.append(np.nan)
            al,lag,amask=attn_last_row(kb, ba)
            al=al.cpu().numpy(); lag=lag.cpu().numpy(); amn=amask.cpu().numpy(); bi=np.digitize(lag, edges[1:-1])
            mass=np.array([sum(al[i][(bi[i]==j)&amn[i]].sum() for i in range(n)) for j in range(len(LAG_BUCKET_LABELS))])/n
            lagm.append(mass)
    lagm=np.array(lagm)

    cf=pd.DataFrame({"sweep_age_yr":sweep,"delta_alpha_norm":dalpha,"h_encoder_shift_l2":hshift,"chm_prob_mean":chm})
    for j,l in enumerate(LAG_BUCKET_LABELS): cf["attn_mass_%s"%l]=lagm[:,j]
    savetable(cf, "counterfactual_age_sweep")

    fig, ax = plt.subplots(2,2,figsize=(12,8))
    ax[0,0].plot(sweep,dalpha,marker="o",color="#0173B2"); ax[0,0].set_title("|| delta_alpha(age) ||")
    ax[0,1].plot(sweep,hshift,marker="o",color="#029E73"); ax[0,1].set_title("mean h_encoder shift vs youngest")
    ax[1,0].plot(sweep,chm,marker="o",color="#CC3311"); ax[1,0].set_title("mean CHM predicted prob")
    for a_ in (ax[0,0],ax[0,1],ax[1,0]): a_.set_xlabel("swept age (yr)")
    im=ax[1,1].imshow(lagm.T,aspect="auto",origin="lower",cmap="magma")
    ax[1,1].set_yticks(range(len(LAG_BUCKET_LABELS))); ax[1,1].set_yticklabels(LAG_BUCKET_LABELS)
    ax[1,1].set_xticks(range(len(sweep))); ax[1,1].set_xticklabels(["%.2g"%a for a in sweep],rotation=45)
    ax[1,1].set_title("attention mass by lag bucket"); ax[1,1].set_xlabel("swept age (yr)")
    fig.colorbar(im,ax=ax[1,1],fraction=.046)
    fig.suptitle("Counterfactual age sweep (events/timestamps fixed) — %s"%MECH_MODEL,y=1.01)
    fig.tight_layout(); savefig(fig, "counterfactual_age_sweep"); plt.show()
    cf

## Outputs

Under `outputs/representation_analysis/`:
- `representations.npz`, `tables/representation_metadata.csv` — both reps + metadata (id, age, band, CHM).
- `tables/diagnostics_unbalanced.csv`, `tables/diagnostics_balanced.csv`, `tables/knn_purity_per_band.csv`.
- `tables/kernel_w_tau_by_age.csv`, `tables/kernel_delta_alpha_by_band.csv`, `tables/attention_mass_by_lag_band.csv`, `tables/counterfactual_age_sweep.csv`.
- `figures/*.png` + `*.svg` for every plot above.

To add **Additive-age / Random-constant**: provide a PIC-fine-tuned ablation checkpoint, set
`enabled=True` + `ckpt_dir`, and complete the ablation branch in `build_model`. **Shuffled-age** is a
checkpoint-free control (set `enabled=True`).